In [228]:
import spacy
from spacy.matcher import PhraseMatcher, Matcher
from spacy.language import Language
from spacy.tokens import Span
from spacy.util import filter_spans
import re
import pandas as pd



In [229]:
nlp=spacy.load('en_core_web_md')
matcher_0 = PhraseMatcher(nlp.vocab, attr="LOWER")
matcher_1 = PhraseMatcher(nlp.vocab, attr="LOWER")
matcher_2 = Matcher(nlp.vocab)
matcher_3 = PhraseMatcher(nlp.vocab, attr="LOWER")
matcher_4 = PhraseMatcher(nlp.vocab, attr="LOWER")
matcher_5 = Matcher(nlp.vocab)


In [230]:
df=pd.read_csv('./saved_files/nlp_dataset/A_Z_medicines_dataset_of_India.csv')

df.head()

,id,name,price(₹),Is_discontinued,manufacturer_name,type,pack_size_label,short_composition1,short_composition2
0,1,Augmentin 625 Duo Tablet,223.42,False,Glaxo SmithKline Pharmaceuticals Ltd,allopathy,strip of 10 tablets,Amoxycillin (500mg),Clavulanic Acid (125mg)
1,2,Azithral 500 Tablet,132.36,False,Alembic Pharmaceuticals Ltd,allopathy,strip of 5 tablets,Azithromycin (500mg),NaN
2,3,Ascoril LS Syrup,118.00,False,Glenmark Pharmaceuticals Ltd,allopathy,bottle of 100 ml Syrup,Ambroxol (30mg/5ml),Levosalbutamol (1mg/5ml)
3,4,Allegra 120mg Tablet,218.81,False,Sanofi India Ltd,allopathy,strip of 10 tablets,Fexofenadine (120mg),NaN
4,5,Avil 25 Tablet,10.96,False,Sanofi India Ltd,allopathy,strip of 15 tablets,Pheniramine (25mg),NaN


In [231]:
list1 = df['short_composition1'].dropna().unique().tolist()
list2 = df['short_composition2'].dropna().unique().tolist()


drug_list = list1+list2

print('drugs name:')
print(drug_list[:5])

print('total nunmber of drugs:',len(drug_list))



drugs name:
['Amoxycillin  (500mg) ', 'Azithromycin (500mg)', 'Ambroxol (30mg/5ml) ', 'Fexofenadine (120mg)', 'Pheniramine (25mg)']
total nunmber of drugs: 11503


In [232]:
pattern=r"\s*\(.*?\)"

for i in range(len(drug_list)):
    drug_list[i] = re.sub(pattern,"", drug_list[i].strip())

print(drug_list[:5])

['Amoxycillin', 'Azithromycin', 'Ambroxol', 'Fexofenadine', 'Pheniramine']


In [233]:
patterns = [nlp.make_doc(med) for med in drug_list]

matcher_0.add("DRUGS", patterns)

In [234]:
df_medicine_list= df['name'].dropna().unique().tolist()

raw_medicine_list= [ i.split() for i in df_medicine_list ]

raw_medicine_list[:5]

[['Augmentin', '625', 'Duo', 'Tablet'],
 ['Azithral', '500', 'Tablet'],
 ['Ascoril', 'LS', 'Syrup'],
 ['Allegra', '120mg', 'Tablet'],
 ['Avil', '25', 'Tablet']]

In [235]:
medicine_list= list(set(map(lambda x : ' '.join(x[:2]),raw_medicine_list)))

print('medicine name:')
print(medicine_list[:5])

print('total medicine:',len(medicine_list))


medicine name:
['Clotrin-B Cream', 'Largy-M Tablet', 'Nirxime 50mg', 'Coxvolt-MR Tablet', 'Zoxicef OX']
total medicine: 232016


In [236]:
patterns = [nlp.make_doc(med) for med in medicine_list]

matcher_1.add("MEDICINE", patterns)

In [237]:
pattern = [
    {"LIKE_NUM": True},
    {"LOWER": {"IN": ["mg", "ml"]}}
]

matcher_2.add("DOSAGE", [pattern])

In [238]:
doc = nlp("Doctor gave me paracetamol 500mg and Dolo 650 for fever")

print("PhraseMatcher Results:")
for match_id, start, end in matcher_0(doc):
    print("drug:", doc[start:end].text)

print("\nPhraseMatcher Results:")
for match_id, start, end in matcher_1(doc):
    print("Medicine:", doc[start:end].text)

print("\nMatcher Results:")
for match_id, start, end in matcher_2(doc):
    print("Dosage:", doc[start:end].text)

PhraseMatcher Results:
drug: paracetamol

PhraseMatcher Results:
Medicine: Dolo 650

Matcher Results:
Dosage: 500mg


In [239]:
disease_dataset = pd.read_csv('./saved_files/nlp_dataset/Disease and symptoms dataset.csv')

disease_name = disease_dataset['diseases'].dropna().unique().tolist()


In [240]:
disease_list = list(map(lambda x: x.lower(),disease_name)) 
print('disease name:')
print(disease_list[:5])
print('total disease:',len(disease_list))

disease name:
['panic disorder', 'vocal cord polyp', 'turner syndrome', 'cryptorchidism', 'poisoning due to ethylene glycol']
total disease: 773


In [241]:
patterns = [nlp.make_doc(med) for med in disease_list]

matcher_3.add("DISEASES", patterns)

In [242]:
add_common_disease = pd.read_csv('./saved_files/nlp_dataset/symptom_dataset.csv')  

add_common_disease = add_common_disease['Symptom_Name'].tolist()



In [243]:
patterns = [nlp.make_doc(med) for med in add_common_disease]

matcher_4.add("SYMPTOM", patterns)

In [244]:
print('symptoms name:')

print(add_common_disease[:5])

print('total symptoms:',len(add_common_disease))

symptoms name:
['Fever', 'High fever', 'Mild fever', 'Low-grade fever', 'Continuous fever']
total symptoms: 227


In [ ]:
pattern_1 = [
    {"LIKE_NUM": True},
    {"LOWER": {"IN": ["day", "days", "week", "weeks", "month", "months"]}}
]

pattern_2 = [
    {"LOWER": {"IN": ["yesterday", "today", "tomorrow"]}}
]

pattern_3 = [
    {"LOWER": "day"},
    {"LOWER": "before"},
    {"LOWER": "yesterday"}
]

pattern_4 = [
    {"LOWER": "last"},
    {"LOWER": {'IN': ['day','month','week','year','morning','afternoon','evening','night']}},

]

matcher_5.add("DURATION", [pattern_1])
matcher_5.add("RELATIVE_DATE", [pattern_2, pattern_3,pattern_4])

In [246]:
matchers = [
    (matcher_0, "DRUG"),
    (matcher_1, "MEDICINE"),
    (matcher_2, "DOSAGE"),
    (matcher_4, "SYMPTOM"),
    (matcher_3, "DISEASE"),
    (matcher_5, "TIME_EXPRESSION"),
]

In [247]:
@Language.component("medical_matcher")
def medicine_words(doc):

    new_ents = list(doc.ents)

    for matcher, label in matchers:
        for _, start, end in matcher(doc):

            span = Span(doc, start, end, label=label)

            new_ents = [
                e for e in new_ents
                if not (e.start < end and e.end > start)
            ]

            new_ents.append(span)

    doc.ents = filter_spans(new_ents)
    return doc

In [248]:
nlp.add_pipe("medical_matcher", after="ner")

print(nlp.pipe_names)

['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'ner', 'medical_matcher']


In [256]:
import dateparser

In [261]:
# doc = nlp("Doctor gave me paracetamol 500mg and Dolo 650 for fever FOR TOMORROW")
# doc=nlp('my sister are feeling pain in legs for 3 days')
# doc=nlp('my sister have cold sore')
# doc=nlp('i am feeling severe pain in legs for 3 days')

def content_details(doc):
    trigger_verbs = ["have",'give', 'take', "suffer", "experience", "diagnose", "feel",'be']

    detail_extract={'patient': None,
    'severity':None,
    'condition':None,
    'disease':None,
    'medicine':None,
    'duration':None,
    'timestamp':None
    }
    for child in doc:
        if (child.dep_ == "nsubj" and child.pos_== 'NOUN') or (child.dep_ == "nsubj" and child.pos_== 'PRON') or (child.dep_ == "dative" and child.text== 'me'):
            detail_extract['patient'] = child.text
                        
        if child.dep_=='amod':
            detail_extract['severity'] = child.text


    for ent in doc.ents:
        if ent.label_ == "MEDICINE":
            detail_extract['medicine'] = ent.text 
        if ent.label_ == "DISEASE":
            detail_extract['disease'] = ent.text
        if ent.label_ == "SYMPTOM":
            detail_extract['condition'] = ent.text
        if ent.label_ == "TIME_EXPRESSION" or ent.label_ == 'DATE':
            detail_extract['duration'] = ent.text

            for token in ent.root.ancestors:
                if token.lemma_ in trigger_verbs:
                    
                    
                    for child in token.children:
                        if (child.dep_ == "nsubj" and child.pos_== 'NOUN') or (child.dep_ == "nsubj" and child.pos_== 'PRON'):
                            detail_extract['patient'] = child.text
                        
                        if child.dep_=='amod':
                            detail_extract['severity'] = child.text

                    
    if(detail_extract['duration']):
        parsed_date = dateparser.parse(detail_extract['duration'])
        formatted_date = parsed_date.strftime("%d/%m/%Y")
        detail_extract['timestamp']=formatted_date

    return detail_extract





In [250]:
disease_dataset[disease_dataset['diseases'].str.contains('cold', case=False)]

,diseases,anxiety and nervousness,depression,shortness of breath,depressive or psychotic symptoms,sharp chest pain,dizziness,insomnia,abnormal involuntary movements,chest tightness,...,stuttering or stammering,problems with orgasm,nose deformity,lump over jaw,sore in nose,hip weakness,back swelling,ankle stiffness or tightness,ankle weakness,neck weakness
86580,cold sore,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
86581,cold sore,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
86582,cold sore,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
86583,cold sore,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
86584,cold sore,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
155209,common cold,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
155210,common cold,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
155211,common cold,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
155212,common cold,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [262]:
demo=[
    "I have had a severe headache for the past 3 days.",

"My stomach has been hurting since last night.",

"I feel dizzy whenever I stand up.",

"I have chest pain and shortness of breath.",

"My child has a fever of 102°F since yesterday.",

"I am experiencing nausea and vomiting after eating.",

"There is swelling and redness around my ankle.",

"I have dry cough and mild fever.",

"My throat is sore and I can't swallow properly.",

"I have frequent urination and burning sensation.",

'i am suffering from fever'
]

In [264]:
for word in demo:
    doc=nlp(word)
    print(doc)
    print(content_details(doc),end='\n\n')

I have had a severe headache for the past 3 days.
{'patient': 'I', 'severity': 'past', 'condition': 'headache', 'disease': None, 'medicine': None, 'duration': '3 days', 'timestamp': '02/03/2026'}

My stomach has been hurting since last night.
{'patient': 'stomach', 'severity': 'last', 'condition': None, 'disease': None, 'medicine': None, 'duration': None, 'timestamp': None}

I feel dizzy whenever I stand up.
{'patient': 'I', 'severity': None, 'condition': 'dizzy', 'disease': None, 'medicine': 'I feel', 'duration': None, 'timestamp': None}

I have chest pain and shortness of breath.
{'patient': 'I', 'severity': None, 'condition': 'shortness of breath', 'disease': None, 'medicine': None, 'duration': None, 'timestamp': None}

My child has a fever of 102°F since yesterday.
{'patient': 'child', 'severity': None, 'condition': 'fever', 'disease': None, 'medicine': None, 'duration': 'yesterday', 'timestamp': '04/03/2026'}

I am experiencing nausea and vomiting after eating.
{'patient': 'I', 's